In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt

from sklearn import linear_model
import lightgbm as lgbm

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [13]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('../data/sample_submission.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)

In [14]:
# Condition target behavior on the sign of the current RET_1 value.
analysis = total_data.copy()
analysis['RET1_SIGN'] = np.select(
    [analysis['RET_1'] > 0, analysis['RET_1'] < 0],
    ['RET_1 > 0', 'RET_1 < 0'],
    default='RET_1 == 0'
)
analysis['TARGET_POSITIVE'] = (analysis['target'] > 0).astype(int)

sign_summary = (
    analysis.groupby('RET1_SIGN', observed=True)
    .agg(
        observations=('target', 'size'),
        mean_ret_1=('RET_1', 'mean'),
        mean_target=('target', 'mean'),
        median_target=('target', 'median'),
        target_std=('target', 'std'),
        target_positive_rate=('TARGET_POSITIVE', 'mean')
    )
    .reindex(['RET_1 < 0', 'RET_1 == 0', 'RET_1 > 0'])
)

print('Target behavior conditioned on RET_1 sign')
print(sign_summary.to_string(float_format=lambda value: f'{value:.6f}'))

# Compare positive versus negative RET_1 directly.
negative_target = analysis.loc[analysis['RET1_SIGN'] == 'RET_1 < 0', 'target']
positive_target = analysis.loc[analysis['RET1_SIGN'] == 'RET_1 > 0', 'target']

if len(negative_target) and len(positive_target):
    target_mean_difference = positive_target.mean() - negative_target.mean()
    target_rate_difference = (
        (positive_target > 0).mean() - (negative_target > 0).mean()
    )
    print(f'\nMean target difference (RET_1 > 0 minus RET_1 < 0): {target_mean_difference:.6f}')
    print(f'Positive-target-rate difference: {target_rate_difference:.6%}')

# Check whether the relationship is consistent across timestamps.
ts_conditioning = (
    analysis[analysis['RET1_SIGN'].isin(['RET_1 < 0', 'RET_1 > 0'])]
    .groupby(['TS', 'RET1_SIGN'], observed=True)
    .agg(
        observations=('target', 'size'),
        mean_target=('target', 'mean'),
        target_positive_rate=('TARGET_POSITIVE', 'mean')
    )
    .reset_index()
    .pivot(index='TS', columns='RET1_SIGN', values='target_positive_rate')
    .dropna()
)
ts_conditioning['positive_minus_negative'] = (
    ts_conditioning['RET_1 > 0'] - ts_conditioning['RET_1 < 0']
)

print('\nPer-TS positive-target-rate difference summary')
print(ts_conditioning['positive_minus_negative'].describe().to_string())
print(
    'Fraction of TS where positive RET_1 has the higher target-positive rate:',
    (ts_conditioning['positive_minus_negative'] > 0).mean()
)

Target behavior conditioned on RET_1 sign
            observations  mean_ret_1  mean_target  median_target  target_std  target_positive_rate
RET1_SIGN                                                                                         
RET_1 < 0  257358.000000   -0.002089    -0.000131      -0.000066    0.003058              0.487978
RET_1 == 0           NaN         NaN          NaN            NaN         NaN                   NaN
RET_1 > 0  269715.000000    0.002146     0.000193       0.000142    0.003143              0.525510

Mean target difference (RET_1 > 0 minus RET_1 < 0): 0.000324
Positive-target-rate difference: 3.753242%

Per-TS positive-target-rate difference summary
count    2522.000000
mean        0.044290
std         0.169919
min        -0.639257
25%        -0.060715
50%         0.040087
75%         0.148590
max         0.764103
Fraction of TS where positive RET_1 has the higher target-positive rate: 0.5951625693893735


In [15]:
# Test whether target agrees with the sign of RET_1.
# Zero returns are kept visible but excluded from the sign-mismatch rate.
sign_test = total_data[['TS', 'RET_1', 'target']].copy()
sign_test['ret_1_sign'] = np.sign(sign_test['RET_1'])
sign_test['target_sign'] = np.sign(sign_test['target'])
sign_test['nonzero_signs'] = (
    sign_test['ret_1_sign'].ne(0) & sign_test['target_sign'].ne(0)
)
sign_test['sign_mismatch'] = np.where(
    sign_test['nonzero_signs'],
    sign_test['ret_1_sign'] != sign_test['target_sign'],
    np.nan
)
sign_test['abs_ret_1'] = sign_test['RET_1'].abs()

sign_mismatch_summary = (
    sign_test.groupby('sign_mismatch', dropna=False)
    .agg(
        observations=('target', 'size'),
        mean_ret_1=('RET_1', 'mean'),
        mean_target=('target', 'mean'),
        median_target=('target', 'median'),
        mean_abs_ret_1=('abs_ret_1', 'mean')
    )
    .rename(index={False: 'same sign', True: 'opposite sign', np.nan: 'at least one zero'})
)

print('RET_1 versus target sign test')
print(sign_mismatch_summary.to_string(float_format=lambda value: f'{value:.6f}'))

valid_signs = sign_test.loc[sign_test['nonzero_signs']].copy()
print(f"\nNonzero sign observations: {len(valid_signs):,} / {len(sign_test):,}")
print(f"Sign-mismatch rate: {valid_signs['sign_mismatch'].mean():.6%}")
print(f"Mean target, same sign: {valid_signs.loc[valid_signs['sign_mismatch'] == False, 'target'].mean():.8f}")
print(f"Mean target, opposite sign: {valid_signs.loc[valid_signs['sign_mismatch'] == True, 'target'].mean():.8f}")

# A useful signal should not depend on only a handful of timestamps.
ts_sign_test = (
    valid_signs.groupby(['TS', 'sign_mismatch'], observed=True)
    .agg(observations=('target', 'size'), mean_target=('target', 'mean'))
    .reset_index()
)
ts_sign_wide = ts_sign_test.pivot(index='TS', columns='sign_mismatch', values='mean_target').dropna()
ts_sign_wide.columns = ['same_sign_mean_target', 'opposite_sign_mean_target']
ts_sign_wide['opposite_minus_same'] = (
    ts_sign_wide['opposite_sign_mean_target'] - ts_sign_wide['same_sign_mean_target']
)

print('\nPer-TS opposite-sign minus same-sign target summary')
print(ts_sign_wide['opposite_minus_same'].describe().to_string())
print(
    'Fraction of TS where opposite-sign observations have the larger mean target:',
    (ts_sign_wide['opposite_minus_same'] > 0).mean()
)
print('\nMismatch rate by TS')
print(
    valid_signs.groupby('TS', observed=True)['sign_mismatch']
    .agg(['size', 'mean'])
    .rename(columns={'size': 'observations', 'mean': 'mismatch_rate'})
    .describe()
)


RET_1 versus target sign test
               observations  mean_ret_1  mean_target  median_target  mean_abs_ret_1
sign_mismatch                                                                      
same sign            273511    0.000111     0.000104       0.000104        0.002169
opposite sign        253562    0.000044    -0.000040      -0.000025        0.002063

Nonzero sign observations: 527,073 / 527,073
Sign-mismatch rate: 48.107568%
Mean target, same sign: 0.00010390
Mean target, opposite sign: -0.00004002

Per-TS opposite-sign minus same-sign target summary
count    2522.000000
mean       -0.000141
std         0.001177
min        -0.008580
25%        -0.000765
50%        -0.000139
75%         0.000516
max         0.005704
Fraction of TS where opposite-sign observations have the larger mean target: 0.4468675654242665

Mismatch rate by TS
       observations  mismatch_rate
count   2522.000000    2522.000000
mean     208.990087       0.476909
std       81.903327       0.088750
min 

In [16]:
# Screen all numeric features for information about target direction and magnitude.
# This is exploratory: rank promising features again on a held-out time split.
feature_target = total_data[['TS', 'target']].join(
    X_train.drop(columns=['TS'], errors='ignore')
).copy()
numeric_features = [
    column for column in X_train.select_dtypes(include=np.number).columns
    if column not in {'target', 'TS'}
]
target_sign = np.sign(feature_target['target'])
feature_rows = []

for feature in numeric_features:
    values = feature_target[feature]
    valid = values.notna() & feature_target['target'].notna()
    nonzero = valid & values.ne(0) & target_sign.ne(0)
    if nonzero.sum() == 0 or values.loc[valid].nunique() < 2:
        continue

    sign_accuracy = (np.sign(values.loc[nonzero]) == target_sign.loc[nonzero]).mean()
    correlation = values.loc[valid].corr(feature_target.loc[valid, 'target'])
    spearman = values.loc[valid].corr(feature_target.loc[valid, 'target'], method='spearman')
    quantiles = values.loc[valid].rank(pct=True)
    lower = feature_target.loc[valid & quantiles.le(0.2), 'target'].mean()
    upper = feature_target.loc[valid & quantiles.ge(0.8), 'target'].mean()
    feature_rows.append({
        'feature': feature,
        'observations': int(valid.sum()),
        'sign_accuracy': sign_accuracy,
        'best_sign_accuracy': max(sign_accuracy, 1 - sign_accuracy),
        'pearson_corr': correlation,
        'spearman_corr': spearman,
        'top20_minus_bottom20_target': upper - lower
    })

feature_screen = pd.DataFrame(feature_rows).set_index('feature')

print(f'Numeric features screened: {len(feature_screen)}')
print('\nBest features by sign accuracy (direction can be inverted)')
print(
    feature_screen.sort_values('best_sign_accuracy', ascending=False)
    [['observations', 'sign_accuracy', 'best_sign_accuracy', 'pearson_corr']]
    .head(20)
    .to_string(float_format=lambda value: f'{value:.6f}')
)
print('\nBest features by absolute Pearson correlation')
print(
    feature_screen.assign(abs_corr=feature_screen['pearson_corr'].abs())
    .sort_values('abs_corr', ascending=False)
    [['observations', 'pearson_corr', 'spearman_corr', 'sign_accuracy', 'top20_minus_bottom20_target']]
    .head(20)
    .to_string(float_format=lambda value: f'{value:.6f}')
)
print('\nBest features by top-minus-bottom target spread')
print(
    feature_screen.reindex(
        feature_screen['top20_minus_bottom20_target'].abs().sort_values(ascending=False).index
    )
    [['observations', 'top20_minus_bottom20_target', 'pearson_corr', 'spearman_corr']]
    .head(20)
    .to_string(float_format=lambda value: f'{value:.6f}')
)

# Test whether feature ranking contains cross-sectional information within each TS.
# A positive value means the top quintile beats the bottom quintile on average.
within_ts_rows = []
for feature in feature_screen.index:
    ranked = feature_target[['TS', feature, 'target']].dropna().copy()
    ranked['feature_percentile'] = ranked.groupby('TS', observed=True)[feature].rank(pct=True)
    top_target = ranked.loc[ranked['feature_percentile'] >= 0.8].groupby('TS', observed=True)['target'].mean()
    bottom_target = ranked.loc[ranked['feature_percentile'] <= 0.2].groupby('TS', observed=True)['target'].mean()
    spread = (top_target - bottom_target).dropna()
    if len(spread):
        within_ts_rows.append({
            'feature': feature,
            'ts_count': len(spread),
            'within_ts_top20_minus_bottom20': spread.mean(),
            'within_ts_positive_fraction': (spread > 0).mean()
        })

within_ts_screen = pd.DataFrame(within_ts_rows).set_index('feature')
print('\nBest cross-sectional features within TS')
print(
    within_ts_screen.reindex(
        within_ts_screen['within_ts_top20_minus_bottom20'].abs().sort_values(ascending=False).index
    )
    .head(20)
    .to_string(float_format=lambda value: f'{value:.8f}')
)


Numeric features screened: 42

Best features by sign accuracy (direction can be inverted)
                       observations  sign_accuracy  best_sign_accuracy  pearson_corr
feature                                                                             
RET_1                        527073       0.518924            0.518924      0.084853
MEDIAN_DAILY_TURNOVER        523435       0.507219            0.507219      0.013405
GROUP                        527073       0.507184            0.507184      0.003950
RET_8                        527073       0.504450            0.504450      0.020715
RET_9                        527073       0.504289            0.504289      0.019591
RET_7                        527073       0.504035            0.504035      0.021213
RET_12                       527067       0.503780            0.503780     -0.001566
SIGNED_VOLUME_1              139567       0.503479            0.503479      0.004278
SIGNED_VOLUME_11             527060       0.496854          

In [17]:
# Among RET_1 sign-mismatch cases, search for another feature that predicts target sign.
# RET_1 itself is excluded because its sign is already known to be wrong in this subset.
mismatch_mask = (
    np.sign(total_data['RET_1']).ne(0)
    & np.sign(total_data['target']).ne(0)
    & (np.sign(total_data['RET_1']) != np.sign(total_data['target']))
)
mismatch_data = total_data.loc[mismatch_mask].copy()
mismatch_target_sign = np.sign(mismatch_data['target'])
mismatch_rows = []

for feature in numeric_features:
    if feature == 'RET_1':
        continue
    values = mismatch_data[feature]
    valid = values.notna()
    nonzero = valid & values.ne(0)
    if nonzero.sum() == 0 or values.loc[valid].nunique() < 2:
        continue
    sign_accuracy = (np.sign(values.loc[nonzero]) == mismatch_target_sign.loc[nonzero]).mean()
    correlation = values.loc[valid].corr(mismatch_data.loc[valid, 'target'])
    mismatch_rows.append({
        'feature': feature,
        'observations': int(valid.sum()),
        'sign_accuracy': sign_accuracy,
        'best_sign_accuracy': max(sign_accuracy, 1 - sign_accuracy),
        'pearson_corr': correlation
    })

mismatch_screen = pd.DataFrame(mismatch_rows).set_index('feature')
print(f'RET_1 sign-mismatch cases: {len(mismatch_data):,}')
print('\nBest features for correcting RET_1 sign-mismatch cases')
print(
    mismatch_screen.sort_values('best_sign_accuracy', ascending=False)
    [['observations', 'sign_accuracy', 'best_sign_accuracy', 'pearson_corr']]
    .head(20)
    .to_string(float_format=lambda value: f'{value:.6f}')
)
print('\nBest correlations inside RET_1 sign-mismatch cases')
print(
    mismatch_screen.assign(abs_corr=mismatch_screen['pearson_corr'].abs())
    .sort_values('abs_corr', ascending=False)
    [['observations', 'pearson_corr', 'sign_accuracy', 'best_sign_accuracy']]
    .head(20)
    .to_string(float_format=lambda value: f'{value:.6f}')
)


RET_1 sign-mismatch cases: 253,562

Best features for correcting RET_1 sign-mismatch cases
                       observations  sign_accuracy  best_sign_accuracy  pearson_corr
feature                                                                             
RET_2                        253562       0.482478            0.517522     -0.049040
SIGNED_VOLUME_1               67793       0.492455            0.507545     -0.029793
GROUP                        253562       0.495283            0.504717      0.010808
MEDIAN_DAILY_TURNOVER        251811       0.495320            0.504680      0.008232
RET_17                       253544       0.495890            0.504110     -0.006841
RET_18                       253539       0.503544            0.503544      0.001699
RET_3                        253562       0.502820            0.502820     -0.009404
SIGNED_VOLUME_15             253539       0.497182            0.502818     -0.008650
RET_5                        253562       0.502812         

In [19]:
# Two-stage model: learn when RET_1 is likely wrong, then learn the correction.
# The split is forward in TS; TS itself is excluded as a feature to avoid time leakage.
from lightgbm import LGBMClassifier

model_features = [
    column for column in X_train.select_dtypes(include=np.number).columns
    if column != 'TS'
]
correction_features = [column for column in model_features if column != 'RET_1']
ordered_ts = np.sort(total_data['TS'].unique())
split_ts = ordered_ts[int(len(ordered_ts) * 0.8)]
train_mask = total_data['TS'] < split_ts
valid_mask = total_data['TS'] >= split_ts

model_X = X_train[model_features].replace([np.inf, -np.inf], np.nan)
train_medians = model_X.loc[train_mask].median()
model_X = model_X.fillna(train_medians)
correction_X = model_X[correction_features]

ret_sign = np.sign(total_data['RET_1']).astype(int)
target_sign = np.sign(total_data['target']).astype(int)
valid_target = target_sign.ne(0) & ret_sign.ne(0)
mismatch_label = (ret_sign != target_sign).astype(int)

mismatch_detector = LGBMClassifier(
    objective='binary', n_estimators=300, learning_rate=0.03,
    num_leaves=31, random_state=42, verbosity=-1
)
mismatch_detector.fit(
    model_X.loc[train_mask],
    mismatch_label.loc[train_mask]
)

correction_train_mask = train_mask & valid_target & mismatch_label.eq(1)
correction_model = LGBMClassifier(
    objective='binary', n_estimators=300, learning_rate=0.03,
    num_leaves=31, random_state=43, verbosity=-1
)
correction_model.fit(
    correction_X.loc[correction_train_mask],
    target_sign.loc[correction_train_mask].eq(1).astype(int)
)

validation = valid_mask & valid_target
baseline_sign = ret_sign.loc[validation].to_numpy()
actual_sign = target_sign.loc[validation].to_numpy()
mismatch_probability = mismatch_detector.predict_proba(model_X.loc[validation])[:, 1]
correction_probability = correction_model.predict_proba(correction_X.loc[validation])[:, 1]

print(f'Forward split: train TS < {split_ts}, validation TS >= {split_ts}')
print(f'Train observations: {train_mask.sum():,}; validation observations: {validation.sum():,}')
print(f'Baseline RET_1 sign accuracy: {(baseline_sign == actual_sign).mean():.6%}')
actual_failures = actual_sign != baseline_sign
correction_sign = np.where(correction_probability >= 0.5, 1, -1)
print(
    'Correction-model accuracy on actual RET_1 failures:',
    (correction_sign[actual_failures] == actual_sign[actual_failures]).mean()
)

print('\nGate threshold results')
threshold_rows = []
for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    apply_correction = mismatch_probability >= threshold
    predicted_sign = baseline_sign.copy()
    predicted_sign[apply_correction] = correction_sign[apply_correction]
    threshold_rows.append({
        'gate_threshold': threshold,
        'corrections_applied_pct': apply_correction.mean(),
        'overall_accuracy': (predicted_sign == actual_sign).mean(),
        'accuracy_on_changed_rows': (
            (predicted_sign[apply_correction] == actual_sign[apply_correction]).mean()
            if apply_correction.any() else np.nan
        )
    })

correction_results = pd.DataFrame(threshold_rows).set_index('gate_threshold')
print(correction_results.to_string(float_format=lambda value: f'{value:.6f}'))


Forward split: train TS < DATE_2018, validation TS >= DATE_2018
Train observations: 422,490; validation observations: 104,583
Baseline RET_1 sign accuracy: 52.101202%
Correction-model accuracy on actual RET_1 failures: 0.5152912524454026

Gate threshold results
                corrections_applied_pct  overall_accuracy  accuracy_on_changed_rows
gate_threshold                                                                     
0.5                            0.121559          0.522265                  0.498938
0.6                            0.000727          0.521022                  0.565789
0.7                            0.000000          0.521012                       NaN
0.8                            0.000000          0.521012                       NaN
0.9                            0.000000          0.521012                       NaN


In [21]:
# Branching model: first use RET_1 sign, then let a separate tree learn exceptions.
# The positive branch learns when a positive RET_1 still leads to a negative target,
# and the negative branch learns the reverse case.
branch_models = {}
for branch_sign in [-1, 1]:
    branch_train = train_mask & valid_target & ret_sign.eq(branch_sign)
    branch_model = LGBMClassifier(
        objective='binary', n_estimators=250, learning_rate=0.03,
        num_leaves=15, min_child_samples=100, random_state=100 + branch_sign,
        verbosity=-1
    )
    branch_model.fit(
        model_X.loc[branch_train],
        target_sign.loc[branch_train].eq(1).astype(int)
    )
    branch_models[branch_sign] = branch_model

branch_validation = valid_mask & valid_target
branch_baseline = ret_sign.loc[branch_validation].to_numpy()
branch_actual = target_sign.loc[branch_validation].to_numpy()
branch_prediction = np.empty(len(branch_actual), dtype=int)

for branch_sign in [-1, 1]:
    branch_rows = branch_baseline == branch_sign
    branch_prediction[branch_rows] = np.where(
        branch_models[branch_sign].predict_proba(
            model_X.loc[branch_validation].iloc[branch_rows]
        )[:, 1] >= 0.5,
        1,
        -1
    )

print('\nBranching tree model')
print(f'Baseline accuracy: {(branch_baseline == branch_actual).mean():.6%}')
print(f'Branch-model accuracy: {(branch_prediction == branch_actual).mean():.6%}')
for branch_sign, branch_name in [(1, 'RET_1 positive'), (-1, 'RET_1 negative')]:
    branch_rows = branch_baseline == branch_sign
    baseline_accuracy = (branch_baseline[branch_rows] == branch_actual[branch_rows]).mean()
    branch_accuracy = (branch_prediction[branch_rows] == branch_actual[branch_rows]).mean()
    failure_rows = branch_rows & (branch_baseline != branch_actual)
    failure_correction_accuracy = (
        (branch_prediction[failure_rows] == branch_actual[failure_rows]).mean()
        if failure_rows.any() else np.nan
    )
    print(
        f'{branch_name}: observations={branch_rows.sum():,}, '
        f'baseline={baseline_accuracy:.6%}, model={branch_accuracy:.6%}, '
        f'actual-failure correction={failure_correction_accuracy:.6%}'
    )



Branching tree model
Baseline accuracy: 52.101202%
Branch-model accuracy: 52.384231%
RET_1 positive: observations=53,611, baseline=53.640111%, model=53.311820%, actual-failure correction=11.450873%
RET_1 negative: observations=50,972, baseline=50.482618%, model=51.408616%, actual-failure correction=26.929477%
